## Likelihood analysis

In the annotation notebook, we have seen that we can add the likelihood as an annotation to each text signal. We can call the same function also to create a likelihood evaluation for each utterance in a sequence and the interations overall. This approach was proposed by:

```Mehri, S., & Eskenazi, M. (2020). USR: An unsupervised and reference free evaluation metric for dialog generation. arXiv preprint arXiv:2005.00456.```

They finetuned a RoBERTa model (USR) specifically to mimic human evaluation, as a reference-free approach. Reference-free means that is does not need a human-created response to compare the system response. Instead it determines the likelihood of the system response given the preceding conversation. The likelihood is calculated by taking the average likelihood of all tokens of the system response according to the model. The USR model was finetuned with conversations from TopicalChat and PersonaChat:

```Karthik Gopalakrishnan, Behnam Hedayatnia, Qinlang Chen, Anna Gottardi, Sanjeev Kwatra, Anu Venkatesh, Raefer Gabriel, Dilek Hakkani-Tur, and Amazon Alexa AI. 2019. Topical-chat: Towards knowledge-grounded open-domain conversations. Proc. Interspeech 2019, pages 1891–1895.```
```Saizheng Zhang, Emily Dinan, Jack Urbanek, Arthur Szlam, Douwe Kiela, and Jason Weston. 2018. Personalizing dialogue agents: I have a dog, do you have pets too? arXiv preprint arXiv:1801.07243.```

Their approach has moderate correlation with human judgments: .42 (TopicalChat) and .48 (PersonaChat). The USR model can be downloaded from the course drive: 

[usr-topicalchat-roberta_ft.zip](https://drive.google.com/file/d/1ODF-trnYeWm_hSkv9-D6-xHlnM47ToBy/view?usp=share_link)

Unpack the zip file and place it anywhere on your local machine. Adapt the path in this notebook below to your local copy of the model:

You can use any other BERT or RoBERTa model (ENCODER) from [hugggingface.co](https://huggingface.co) to score the likelihood. 


Each system token of the uterance under consideration is turned into a masked token to predict the most probable tokens according to the model (cut-off by ```len_top_tokens```). If the system token is in the list, it will receive the score from the model. If it is not in the list, the token scores ```0```.

## Prerequisites

This notebooks relies on the transformers package and the EMISSOR package for loading the EMISSOR scenarios. These packages can be installed through ```pip```:

In [12]:
#!pip install emissor
#!pip install numpy==1.26.3
# !pip install torch==2.2.2
# !pip install transformers==4.51.3

## Loading an ENCODER model for the masked-task from huggingface or disk

In [13]:
import re
import numpy
from transformers import pipeline, AutoTokenizer

model_name = "google-bert/bert-base-uncased"
#model_name = "FacebookAI/xlm-roberta-base"
#model_name = "FacebookAI/roberta-base"

# Path to your local copy of the USR model
model_name ="../../models/usr-topicalchat-roberta_ft"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = pipeline("fill-mask", model=model_name)

Device set to use mps:0


In [14]:
def mask_target_sentence(context, target):
    masked_targets = []
    ## We limit the length of the target as too long utterance break the token limit
    target_tokens = re.split(' ', target[:500])
    for index, token in enumerate(target_tokens):
        sequence = context + " "
        for token in target_tokens[:index]:
            sequence += token + " "
        sequence += tokenizer.mask_token
        for token in target_tokens[index + 1:]:
            sequence += " " + token
        masked_targets.append(sequence)
    return masked_targets, target_tokens

def sentence_likelihood(context, target):
    masked_targets, target_tokens = mask_target_sentence(context, target)
    expected_target = ""
    max_scores = []
    scores = []
    for masked_target, token in zip(masked_targets, target_tokens):
        results = model(masked_target)
        expected_target += results[0]['token_str'] + " "
        max_scores.append(results[0]['score'])
        match = False
        for result in results:
            if result['token_str'].lower().strip() == token.lower():
                scores.append(result['score'])
                match = True
                break
        if not match:
            scores.append(0)
    likelihood = sum(scores) / len(scores)
    max_likelihood = sum(max_scores) / len(max_scores)

    return likelihood, expected_target, max_likelihood

def score_pairs_for_likelihood(turns: []):
    for context, target in turns:
        llh, best_sentence, max_score = sentence_likelihood(context, target)
        print('Likelihood:', llh, 'Max score:', max_score, 'Best sentence:', best_sentence)

## Obtaining the conversations from EMISSOR

The next function gets the text signals from a conversation captured in an EMISSOR scenario.

In [15]:
import os
from emissor.persistence import ScenarioStorage
from emissor.representation.scenario import Modality
from emissor.representation.scenario import Signal, TextSignal

In [16]:
def get_text_signals_from_a_scenario(emissor_folder:str, scenario_id:str):
    text_signals=[]
    scenario_folder = os.path.join(emissor_folder, scenario_id)
    scenario_storage = ScenarioStorage(emissor_folder)
    scenario_ctrl = scenario_storage.load_scenario(scenario_id)
    try:
        text_signals = scenario_ctrl.get_signals(Modality.TEXT)
    except:
        print('Error loading text signals from text.json')
    return text_signals


def get_speaker_from_text_signal(textSignal: TextSignal):
    speaker = None
    mentions = textSignal.mentions
    for mention in mentions:
        annotations = mention.annotations
        for annotation in annotations:
            if annotation.type == 'ConversationalAgent':
                speaker = annotation.value
                break
        if speaker:
            break
    return speaker


In [17]:
EMISSOR="./data/emissor"
SCENARIO="14a1c27d-dfd2-465b-9ab2-90e9ea91d214"

text_signals = get_text_signals_from_a_scenario(EMISSOR, SCENARIO)
llh_results = []
target = ""
context= ""
for index, text_signal in enumerate(text_signals):
    print(f"Processing turn {index}/{len(text_signals) - 1}")
    speaker = get_speaker_from_text_signal(text_signal)
    if index==0:
        content = ""
    else:
        context = target
    target = text_signal.text
    llh, model_sentence, max_score = sentence_likelihood(context, target)
    llh_dict = {"speaker": speaker, "llh": llh, "turn": target, "model_turn": model_sentence, "max_llh": max_score}
    llh_results.append(llh_dict)

Processing turn 0/33
Processing turn 1/33
Processing turn 2/33
Processing turn 3/33
Processing turn 4/33
Processing turn 5/33
Processing turn 6/33
Processing turn 7/33
Processing turn 8/33
Processing turn 9/33
Processing turn 10/33
Processing turn 11/33
Processing turn 12/33
Processing turn 13/33
Processing turn 14/33
Processing turn 15/33
Processing turn 16/33
Processing turn 17/33
Processing turn 18/33
Processing turn 19/33
Processing turn 20/33
Processing turn 21/33
Processing turn 22/33
Processing turn 23/33
Processing turn 24/33
Processing turn 25/33
Processing turn 26/33
Processing turn 27/33
Processing turn 28/33
Processing turn 29/33
Processing turn 30/33
Processing turn 31/33
Processing turn 32/33
Processing turn 33/33


In [18]:
speaker_llh = []
agent_llh = []
speaker_max_llh = []
agent_max_llh = []
for llh in llh_results:
    print(llh["turn"], ":", llh["model_turn"])
    if llh["speaker"]=="LEOLANI":
        agent_llh.append(llh["llh"])
        agent_max_llh.append(llh["max_llh"])
    else:
        speaker_llh.append(llh["llh"])
        speaker_max_llh.append(llh["max_llh"])

average_speaker_llh = sum(speaker_llh)/len(speaker_llh)
average_speaker_max_llh = sum(speaker_max_llh)/len(speaker_max_llh)
average_agent_llh = sum(agent_llh)/len(agent_llh)
average_agent_max_llh = sum(agent_max_llh)/len(agent_max_llh)

print('average_speaker_llh',average_speaker_llh)
print('average_speaker_max_llh',average_speaker_max_llh)
print('average_agent_llh',average_agent_llh)
print('average_agent_max_llh',average_agent_max_llh)

Lucy drinks wine :  and  of  ? 
Sorry, I did not get that. :  .  i  do  not  know  that 
Lucy is my friend : she  is  my  favorite 
I see. This is what I got from what you said:  (wilbur, get, that) (wilbur, get, wine) (wilbur, drink, wine) (wilbur, be, wilbur-'-s-friend)  :  you  think  this  is  what  i  got  from  what  you  said  wine  (  do  wine  (  drink  that  (  get  )  ,  '  that  . 
I know. I have heard about you before :  i  think i  have  heard  of  you  . 
I know Lucy :  you  like  . 
I see. Cannot make much of what you said. :  i  did  you  remember  sense  of  what  he  say 
Fred knows Lucy too : everyone  and  that  . 
Ok, so:  (wilbur, know, lucy) (wilbur, know, lucy) (wilbur, can-make-much-of, lucy-what-leolani-said) (wilbur, can-make-much-of, lucy)  :  .  " Fred  (  know    (  know    (  tell    (  know  and  ) 
Who knows Lucy? :  ,  is  ? 
Ok, so:  (, know, lucy) (, know, lucy)  :  or  Lucy  Lucy  ,  like  )  (  know  )  ? 
I am out of words. :  i  ran  afraid  of 

## End of notebook